# AI Personal Trainer on IBM watsonx.ai

> **Not medical advice.** Educational project; check with a qualified professional before starting a new program.

Photos of the gym equipment you have → a watsonx.ai vision model lists the equipment → a watsonx.ai
text model writes a weekly plan that uses only that equipment → the notebook checks the plan against
the equipment list and exports an HTML report.

**Setup:** `WATSONX_APIKEY`, `WATSONX_PROJECT_ID` and `WATSONX_URL` in the repo's `.env`, and one or more
equipment photos in `data/gym/` (jpg, png or webp). Models can be changed with `WATSONX_TEXT_MODEL` and
`WATSONX_VISION_MODEL`; IBM retires models regularly, so check what your region offers.

In [22]:
# Cell 1 — Settings and watsonx.ai connection
import os, re, json, base64
from datetime import date
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference

load_dotenv(find_dotenv(usecwd=True))
PROJECT_DIR = Path.cwd().parent if Path.cwd().name in ("notebooks", "src") else Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

missing = [k for k in ("WATSONX_APIKEY", "WATSONX_PROJECT_ID", "WATSONX_URL") if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Set {', '.join(missing)} in the repo's .env (see .env.example)")

TEXT_MODEL = os.getenv("WATSONX_TEXT_MODEL", "meta-llama/llama-3-3-70b-instruct")
VISION_MODEL = os.getenv("WATSONX_VISION_MODEL", "meta-llama/llama-4-maverick-17b-128e-instruct-fp8")
credentials = Credentials(url=os.environ["WATSONX_URL"], api_key=os.environ["WATSONX_APIKEY"])

def watsonx_model(model_id, **params):
    return ModelInference(model_id=model_id, credentials=credentials,
                          project_id=os.environ["WATSONX_PROJECT_ID"], params=params)

def reply_text(resp) -> str:
    return (resp.get("choices", [{}])[0].get("message", {}).get("content") or "").strip()

print(f"Region: {os.environ['WATSONX_URL']}\nText model: {TEXT_MODEL}\nVision model: {VISION_MODEL}")

Region: https://us-south.ml.cloud.ibm.com
Text model: meta-llama/llama-3-3-70b-instruct
Vision model: meta-llama/llama-4-maverick-17b-128e-instruct-fp8


In [23]:
# Cell 2 — Load the equipment photos
GYM_DIR = PROJECT_DIR / "data" / "gym"
paths = sorted(p for p in GYM_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
if not paths:
    raise FileNotFoundError(f"Put one or more photos of gym equipment in {GYM_DIR}")

import io
from PIL import Image

def to_jpeg_b64(path, max_side=1024):
    """Shrink to at most 1024 px and re-encode as JPEG: a smaller upload is faster and more reliable."""
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()

images = [(p.name, "image/jpeg", to_jpeg_b64(p)) for p in paths]
print(f"Loaded {len(images)} photo(s):", [name for name, _, _ in images],
      f"| upload size {sum(len(b) for _, _, b in images) / 1e6:.1f} MB")

Loaded 3 photo(s): ['gym equip.png', 'gym equip2.png', 'gym equip3.png'] | upload size 0.2 MB


In [24]:
# Cell 3 — Vision model: what equipment is in each photo?
VISION_PROMPT = (
    "You are a careful personal-trainer assistant. List the gym equipment clearly visible in this photo. "
    "Don't guess at things you can't see. Reply with JSON only, in this form: "
    '{"equipment": ["adjustable dumbbells", "flat bench"], "category": "strength_training", "workout_type": "upper_body"}. '
    "category is one of strength_training, cardio, flexibility, other. "
    "workout_type is one of upper_body, lower_body, full_body, core, cardio."
)

def parse_json(text: str) -> dict:
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    match = re.search(r"\{.*\}", text, flags=re.S)
    if not match:
        raise ValueError(f"No JSON in the model's reply:\n{text}")
    return json.loads(match.group(0))

vision = watsonx_model(VISION_MODEL, max_tokens=300, temperature=0)
gym_equipment = []
def chat_with_retry(model, messages, attempts=3):
    """Retry brief network failures (dropped or reset connections); other errors stop immediately."""
    import time, httpx
    for attempt in range(1, attempts + 1):
        try:
            return model.chat(messages=messages)
        except (httpx.TransportError, ConnectionError) as e:
            if attempt == attempts:
                raise
            print(f"Network error ({e.__class__.__name__}), retrying {attempt}/{attempts - 1}...")
            time.sleep(2 * attempt)

for name, mime, b64 in images:
    resp = chat_with_retry(vision, [{"role": "user", "content": [
        {"type": "text", "text": VISION_PROMPT},
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
    ]}])
    raw = reply_text(resp)
    found = parse_json(raw)
    gym_equipment.append({"filename": name, "description": ", ".join(found.get("equipment", [])),
                          "category": found.get("category", ""), "workout_type": found.get("workout_type", "")})
    print(f"{name}: {raw}")

C:\Users\sidhu\Documents\VSCodeProjects\ds-portfolio\project-17-health-fitness-assistant\.venv\Lib\site-packages\ibm_watsonx_ai\wml_resource.py:100: WatsonxAPIWarning: This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.
ID: disclaimer_warning
More info: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)


gym equip.png: {"equipment": ["adjustable bench", "barbell", "dumbbells"], "category": "strength_training", "workout_type": "upper_body"}
gym equip2.png: {"equipment": ["seated row machine"], "category": "strength_training", "workout_type": "upper_body"}
gym equip3.png: {"equipment": ["rowing machine"], "category": "cardio", "workout_type": "full_body"}


In [25]:
# Cell 4 — Detected equipment, with a human check before planning
# Vision models can misread machines: in one run Llama 4 Maverick called a multi-press
# (chest and shoulder press) a "seated row machine". Correct any mistakes here; the
# original reading is kept in the saved run.
CORRECTIONS = {
    "gym equip2.png": "multi-press machine (chest press and shoulder press)",
}
for item in gym_equipment:
    if item["filename"] in CORRECTIONS:
        item["vision_said"] = item["description"]
        item["description"] = CORRECTIONS[item["filename"]]
df_equipment = pd.DataFrame(gym_equipment)
df_equipment

,filename,description,category,workout_type,vision_said
0,gym equip.png,"adjustable bench, barbell, dumbbells",strength_training,upper_body,NaN
1,gym equip2.png,multi-press machine (chest press and shoulder ...,strength_training,upper_body,seated row machine
2,gym equip3.png,rowing machine,cardio,full_body,NaN


In [26]:
# -----------------------------------------------------------
# Cell 5: Define the USER PROFILE (who the plan is for)
# -----------------------------------------------------------
# The workout plan depends heavily on:
#   - age
#   - fitness level
#   - training experience
#   - personal goals
#   - available days per week
#   - injuries or limitations
#   - exercise preferences
#
# This block allows students to MODIFY these inputs and see how
# the planner model changes the weekly workout plan.
#
# This is an excellent place for experimentation:
#   - Change goals ("weight loss", "hypertrophy", "flexibility")
#   - Change availability (2 vs 5 days per week)
#   - Add injuries ("knee pain", "no overhead lifting")
#   - Try different preference sets
#
# The planner model adapts the plan to these inputs.
# -----------------------------------------------------------

user_profile = {
    "name": "Gurveer",                         # User's name (only for display)
    "age": 28,                              # Age can influence exercise intensity
    "gender": "Male",                # Optional metadata

    # Beginner / intermediate / advanced
    # This will affect volume, exercise difficulty, and selection.
    "fitness_level": "intermediate",

    # Experience with lifting influences safety + complexity of movements.
    "experience_with_strength_training": "moderate",  # none / low / moderate / high

    # A list of goals; the model will integrate these into the plan structure.
    "primary_goals": [
        "build strength",
        "improve general fitness"
    ],

    # Number of workout days the plan should cover.
    "days_per_week_available": 4,

    # A realistic session length helps ensure the plan isn’t too long.
    "session_length_minutes": 50,

    # List any injuries—model avoids unsafe exercises when possible.
    "injuries_or_limitations": [],

    # Personal likes and dislikes guide exercise selection.
    "exercise_preferences": {
        "likes": ["free weights"],
        "dislikes": ["high impact jumping"]
    }
}

# -----------------------------------------------------------
# Convert both the user profile and equipment list into JSON
# -----------------------------------------------------------
# The planner model receives these values AS TEXT in the prompt.
# JSON formatting ensures:
#   - clean structure
#   - predictable formatting
#   - easier parsing for LLMs
# -----------------------------------------------------------

user_profile_json = json.dumps(user_profile, indent=2)
equipment_json = json.dumps(gym_equipment, indent=2)

# (Optional for debugging)
print("User profile (JSON):")
print(user_profile_json)

print("\nEquipment list (JSON):")
print(equipment_json)

User profile (JSON):
{
  "name": "Gurveer",
  "age": 28,
  "gender": "Male",
  "fitness_level": "intermediate",
  "experience_with_strength_training": "moderate",
  "primary_goals": [
    "build strength",
    "improve general fitness"
  ],
  "days_per_week_available": 4,
  "session_length_minutes": 50,
  "injuries_or_limitations": [],
  "exercise_preferences": {
    "likes": [
      "free weights"
    ],
    "dislikes": [
      "high impact jumping"
    ]
  }
}

Equipment list (JSON):
[
  {
    "filename": "gym equip.png",
    "description": "adjustable bench, barbell, dumbbells",
    "category": "strength_training",
    "workout_type": "upper_body"
  },
  {
    "filename": "gym equip2.png",
    "description": "multi-press machine (chest press and shoulder press)",
    "category": "strength_training",
    "workout_type": "upper_body",
    "vision_said": "seated row machine"
  },
  {
    "filename": "gym equip3.png",
    "description": "rowing machine",
    "category": "cardio",
    "w

In [27]:
# Cell 6 — Text model: weekly plan using only the detected equipment
COLUMNS = ["Day", "Focus", "Exercise", "Equipment", "Sets", "RepsOrTime", "TargetMuscle",
           "Intensity", "Warmup", "Cooldown", "DayNotes"]
days = user_profile.get("days_per_week_available", 3)
minutes = user_profile.get("session_length_minutes", 45)
system = (
    "You are an experienced, safety-conscious personal trainer. Use only the equipment listed, or bodyweight. "
    "Respect injuries and dislikes. Use each machine only for the movements it is built for; "
    "where the equipment list names a machine's movements in brackets, use it only for those. "
    f"Plan exactly {days} training days and don't list rest days. Each training day has 4 to 6 exercises that fit "
    f"a {minutes}-minute session. Across the week, train upper body, lower body and core. "
    "Reply with a pipe-delimited table only: no explanations, no code fences, "
    "and no '|' inside any value. The first row is the header: " + " | ".join(COLUMNS)
)
# Send only the corrected equipment: the vision model's original reading stays out of the prompt.
planner_equipment = [{k: v for k, v in e.items() if k != "vision_said"} for e in gym_equipment]
user = f"USER PROFILE:\n{user_profile_json}\n\nAVAILABLE EQUIPMENT:\n{json.dumps(planner_equipment, indent=2)}"

planner = watsonx_model(TEXT_MODEL, max_tokens=1500, temperature=0.2)
plan_text = reply_text(chat_with_retry(planner, [{"role": "system", "content": system},
                                                  {"role": "user", "content": user}]))
if "|" not in plan_text:
    raise ValueError(f"The planner didn't return a table:\n{plan_text}")
print(plan_text)

Day | Focus | Exercise | Equipment | Sets | RepsOrTime | TargetMuscle | Intensity | Warmup | Cooldown | DayNotes
Monday | Upper Body | Bench Press | adjustable bench, barbell | 3 | 8-12 | Chest, Triceps, Shoulders | Moderate to High | 5 minutes of cardio and dynamic stretching | 5 minutes of stretching | Focus on proper form and engage core
Monday | Upper Body | Dumbbell Rows | dumbbells | 3 | 10-15 | Back, Biceps | Moderate |  |  | Alternate arms and keep back straight
Monday | Upper Body | Dumbbell Shoulder Press | dumbbells | 3 | 10-12 | Shoulders | Moderate |  |  | Keep core engaged and avoid arching back
Monday | Upper Body | Tricep Dips | adjustable bench | 3 | 12-15 | Triceps | Moderate |  |  | Lower body down slowly and push back up
Tuesday | Lower Body | Squats | bodyweight | 3 | 8-12 | Legs, Glutes | Moderate to High | 5 minutes of cardio and dynamic stretching | 5 minutes of stretching | Keep back straight and engage core
Tuesday | Lower Body | Calf Raises | bodyweight | 3 |

C:\Users\sidhu\Documents\VSCodeProjects\ds-portfolio\project-17-health-fitness-assistant\.venv\Lib\site-packages\ibm_watsonx_ai\wml_resource.py:100: WatsonxAPIWarning: This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.
ID: disclaimer_warning
More info: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)


In [28]:
# -----------------------------------------------------------
# Cell 7: Parse the workout plan → Create DataFrame → Build HTML report
# -----------------------------------------------------------
# This is the final “assembly” cell. It does FOUR major things:
#
# 1.  SAFELY parse the raw planner output (plan_text)
#     The model returns plain text, so we:
#        - split into lines
#        - detect the header row
#        - detect the exercise rows
#        - handle formatting mistakes
#
# 2.  Convert the structured rows into a Pandas DataFrame
#     This becomes df_plan, which is our final “clean table”.
#
# 3.  Build a readable HTML report with:
#        - a header block (user profile summary)
#        - a section for each day with an exercise table
#
# 4.  Save the final HTML file ("workout_plan.html") to disk
#     You can open it in any browser → Print → Save as PDF.
# -----------------------------------------------------------

import pandas as pd
import os

# -----------------------------
# 1) Parse planner output (plan_text) into df_plan
# -----------------------------

# Safety check: make sure the planner produced real text
if not isinstance(plan_text, str) or not plan_text.strip():
    raise ValueError(
        "plan_text is empty or not a string.\n"
        "Make sure the planner_model cell ran correctly and returned text."
    )

# Split output into individual non-empty lines
lines = [l.strip() for l in plan_text.splitlines() if l.strip()]

# Keep only lines that contain the pipe character '|'
# This ensures we ignore any accidental text the model might add.
table_lines = [l for l in lines if "|" in l]

if not table_lines:
    raise ValueError(
        "No table-like lines found in the plan_text.\n"
        "Check the raw planner output above."
    )

# The first line is the header with column names
header_line = table_lines[0]

# Everything after the header is a data row
data_lines = table_lines[1:]

# Expected column names (exact order)
expected_cols = [
    "Day", "Focus", "Exercise", "Equipment", "Sets",
    "RepsOrTime", "TargetMuscle", "Intensity",
    "Warmup", "Cooldown", "DayNotes"
]

# Show the parsed header so students can understand what’s going on
header_parts = [p.strip() for p in header_line.split("|")]
print("Parsed header:", header_parts)

rows = []

# -----------------------------
# Parse each line of the table
# -----------------------------
for line in data_lines:
    parts = [p.strip() for p in line.split("|")]

    # If too many columns → merge extras into the last one
    if len(parts) > len(expected_cols):
        parts = parts[:len(expected_cols)-1] + [" ".join(parts[len(expected_cols)-1:])]

    # If too few columns → pad with empty fields
    if len(parts) < len(expected_cols):
        parts += [""] * (len(expected_cols) - len(parts))

    # If any mismatch remains → skip the line
    if len(parts) != len(expected_cols):
        print("⚠️ Skipping malformed line:", line)
        continue

    # Build a dictionary: {"Day": "..", "Focus": "..", ...}
    row = dict(zip(expected_cols, parts))
    rows.append(row)

# Convert to DataFrame
df_plan = pd.DataFrame(rows)

# Remove completely empty rows (if any slipped in)
df_plan = df_plan.replace("", pd.NA).dropna(how="all").fillna("")

print("\n✅ Parsed workout plan into DataFrame:")
print(df_plan)
print("Shape:", df_plan.shape)

if df_plan.empty:
    raise ValueError("df_plan is empty. The plan text may not have followed the expected table format.")


# -----------------------------------------------------------
# 2) Build the HTML report (header + daily tables)
# -----------------------------------------------------------

# Remove accidental whitespace in column names
df_plan = df_plan.rename(columns=lambda c: c.strip() if isinstance(c, str) else c)

print("\nFinal df_plan columns:", list(df_plan.columns))

# Create a header block summarizing the user's profile
goals_text = ", ".join(user_profile.get("primary_goals", []))

header_html = f"""
<h1>Personalized Workout Plan</h1>
<p><strong>Name:</strong> {user_profile.get("name", "N/A")}</p>
<p><strong>Age:</strong> {user_profile.get("age", "N/A")}</p>
<p><strong>Gender:</strong> {user_profile.get("gender", "N/A")}</p>
<p><strong>Fitness Level:</strong> {user_profile.get("fitness_level", "N/A")}</p>
<p><strong>Goals:</strong> {goals_text}</p>
<p><strong>Days/Week Available:</strong> {user_profile.get("days_per_week_available", "N/A")}</p>
<p><strong>Session Length:</strong> {user_profile.get("session_length_minutes", "N/A")} minutes</p>
"""

if user_profile.get("injuries_or_limitations"):
    header_html += (
        f"<p><strong>Injuries / Limitations:</strong> "
        f"{', '.join(user_profile['injuries_or_limitations'])}</p>"
    )

# Start the HTML output
html_output = [header_html, "<hr>"]

# Columns to show in each day's exercise table
preferred_cols = [
    "Exercise", "Equipment", "Sets", "RepsOrTime", "TargetMuscle", "Intensity"
]

# Safety check: ensure we have a valid Day column
if "Day" not in df_plan.columns:
    df_plan["Day"] = "Plan"

# Loop through each day and build its section
for day_name in df_plan["Day"].unique():
    day_df = df_plan[df_plan["Day"] == day_name].copy()

    focus = day_df["Focus"].iloc[0] if "Focus" in day_df.columns else ""
    warmup = day_df["Warmup"].iloc[0] if "Warmup" in day_df.columns else ""
    cooldown = day_df["Cooldown"].iloc[0] if "Cooldown" in day_df.columns else ""
    notes = day_df["DayNotes"].iloc[0] if "DayNotes" in day_df.columns else ""

    # Keep only the columns we want to show (if they exist)
    display_cols = [c for c in preferred_cols if c in day_df.columns]

    html_output.append(f"<h2>{day_name}</h2>")
    
    if focus:
        html_output.append(f"<p><strong>Focus:</strong> {focus}</p>")
    if warmup:
        html_output.append(f"<p><strong>Warmup:</strong> {warmup}</p>")

    # Insert table of exercises
    if display_cols:
        html_output.append(day_df[display_cols].to_html(index=False, border=1))
    else:
        html_output.append(day_df.to_html(index=False, border=1))

    if cooldown:
        html_output.append(f"<p><strong>Cooldown:</strong> {cooldown}</p>")
    if notes:
        html_output.append(f"<p><strong>Notes:</strong> {notes}</p><br>")

# Join all blocks into a single HTML document
full_html = "\n".join(html_output)

# Save the HTML to disk
output_path = str(OUTPUT_DIR / "workout_plan.html")
with open(output_path, "w", encoding="utf-8") as f:
    f.write(full_html)

print(f"\n✅ Saved HTML to: {os.path.abspath(output_path)}")
print("Open this file in a web browser, then go to: File → Print → Save as PDF")
from IPython.display import HTML, display
display(HTML(full_html))

Parsed header: ['Day', 'Focus', 'Exercise', 'Equipment', 'Sets', 'RepsOrTime', 'TargetMuscle', 'Intensity', 'Warmup', 'Cooldown', 'DayNotes']

✅ Parsed workout plan into DataFrame:
          Day       Focus                 Exercise  \
0      Monday  Upper Body              Bench Press   
1      Monday  Upper Body            Dumbbell Rows   
2      Monday  Upper Body  Dumbbell Shoulder Press   
3      Monday  Upper Body              Tricep Dips   
4     Tuesday  Lower Body                   Squats   
5     Tuesday  Lower Body              Calf Raises   
6     Tuesday  Lower Body            Glute Bridges   
7     Tuesday  Lower Body                 Step Ups   
8   Wednesday  Upper Body              Chest Press   
9   Wednesday  Upper Body           Shoulder Press   
10  Wednesday  Upper Body              Bicep Curls   
11  Wednesday  Upper Body        Tricep Extensions   
12   Thursday   Full Body                   Rowing   
13   Thursday        Core                    Plank   
14   Thur

Exercise,Equipment,Sets,RepsOrTime,TargetMuscle,Intensity
Bench Press,"adjustable bench, barbell",3,8-12,"Chest, Triceps, Shoulders",Moderate to High
Dumbbell Rows,dumbbells,3,10-15,"Back, Biceps",Moderate
Dumbbell Shoulder Press,dumbbells,3,10-12,Shoulders,Moderate
Tricep Dips,adjustable bench,3,12-15,Triceps,Moderate
Exercise,Equipment,Sets,RepsOrTime,TargetMuscle,Intensity
Squats,bodyweight,3,8-12,"Legs, Glutes",Moderate to High
Calf Raises,bodyweight,3,12-15,Calves,Moderate
Glute Bridges,bodyweight,3,10-12,"Glutes, Hamstrings",Moderate
Step Ups,bodyweight,3,10-12 per leg,"Legs, Glutes",Moderate
Exercise,Equipment,Sets,RepsOrTime,TargetMuscle,Intensity


In [29]:
# Cell 8 — Check: does every exercise use equipment we actually have?
# Strict on purpose: every specific word in the plan's equipment must appear in what was detected,
# so "seated row machine" doesn't pass just because a "rowing machine" was detected.
GENERIC = {"machine", "equipment", "adjustable", "weight", "set", "and", "with", "the"}
NO_EQUIPMENT = {"none", "bodyweight", "body weight", "n/a", "", "floor", "mat"}

def words(text: str) -> set:
    found = re.findall(r"[a-z]+", text.lower())
    singular = {w[:-1] if w.endswith("s") and not w.endswith("ss") else w for w in found if len(w) > 2}
    return singular - GENERIC

detected_words = words(" ".join(df_equipment["description"]))

def uses_known_equipment(equipment: str) -> bool:
    if equipment.strip().lower() in NO_EQUIPMENT:
        return True
    needed = words(equipment)
    return bool(needed) and needed <= detected_words

exercises = df_plan[df_plan["Exercise"].str.strip() != ""].copy()
exercises["equipment_ok"] = exercises["Equipment"].map(uses_known_equipment)
flagged = exercises.loc[~exercises["equipment_ok"], ["Day", "Exercise", "Equipment"]]

# Machines with their movements named in brackets, e.g. "multi-press machine (chest press and shoulder press)":
# an exercise on that machine must be one of those movements (a "seated row" on a press machine is not).
machine_moves = {}
for item in df_equipment["description"]:
    for name, moves in re.findall(r"([^,(]+)\(([^)]*)\)", item):
        machine_moves[frozenset(words(name))] = words(moves)

def fits_machine(row) -> bool:
    for machine, moves in machine_moves.items():
        if machine and machine <= words(row["Equipment"]):
            return bool(words(row["Exercise"]) & moves)
    return True

exercises["movement_ok"] = exercises.apply(fits_machine, axis=1)
wrong_movement = exercises.loc[~exercises["movement_ok"], ["Day", "Exercise", "Equipment"]]

per_day = exercises.groupby("Day", sort=False).size()
muscles = " ".join(exercises["TargetMuscle"].str.lower())
checks = {
    "training_days": f"{len(per_day)} planned, {days} requested",
    "thin_days": {d: int(n) for d, n in per_day.items() if n < 4},
    "trains_lower_body": bool(re.search(r"leg|quad|hamstring|glute|calf|lower", muscles)),
    "trains_core": bool(re.search(r"core|ab|oblique", muscles)),
    "exercises_using_undetected_equipment": flagged.to_dict("records"),
    "exercises_a_machine_cannot_do": wrong_movement.to_dict("records"),
}
print(f"{exercises['equipment_ok'].sum()} of {len(exercises)} exercises use detected equipment or bodyweight.")
print(json.dumps(checks, indent=2))

16 of 16 exercises use detected equipment or bodyweight.
{
  "training_days": "4 planned, 4 requested",
  "thin_days": {},
  "trains_lower_body": true,
  "trains_core": true,
  "exercises_using_undetected_equipment": [],
  "exercises_a_machine_cannot_do": []
}


In [30]:
# Cell 9 — Save the run, labelled with the models that produced it
run = {"date": date.today().isoformat(), "region": os.environ["WATSONX_URL"],
       "text_model": TEXT_MODEL, "vision_model": VISION_MODEL,
       "equipment": gym_equipment, "exercises": len(exercises), "checks": checks}
json.dump(run, open(OUTPUT_DIR / "trainer_run.json", "w"), indent=2)
df_plan.to_csv(OUTPUT_DIR / "workout_plan.csv", index=False)
print("Saved outputs/trainer_run.json, workout_plan.csv and workout_plan.html")

Saved outputs/trainer_run.json, workout_plan.csv and workout_plan.html
